# Predicting Hit Songs on Spotify — Experiments

This notebook reproduces the experiments and tables reported in the final paper. It is a thin wrapper around the scripts in `src/`; run cells in order from a fresh kernel started at the repo root.

**Team:** Charlotte Lin · Irene Wu · Jay Huang · Jessica Hsiao · Zoe Tseng (INFO 5368 · Cornell Tech)

**Repo:** https://github.com/Irene-Wu-1002/INFO_5368_Final_Project

**What this notebook produces:**
- Table I (stratified random split) → `src/train.py`
- Table II (temporal split, 2025 cutoff) → `src/train.py`
- Table III (chart-context benchmark) → `src/train.py`
- Table IV (feature-block ablation) → `src/eval_ablation.py`
- Artist-stratified 5-fold evaluation (§III.D paragraph) → `src/eval_artist_stratified.py`

All metrics in the paper are loaded from `artifacts/metrics.json` (production / temporal / benchmark) and `artifacts/ablation_metrics.json` (ablation). Re-running `train.py` and the eval scripts regenerates those files.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Make src/ importable so we can call the same code train.py / eval_*.py use
ROOT = Path.cwd()
if (ROOT / "src").is_dir():
    sys.path.insert(0, str(ROOT / "src"))
else:
    # Notebook was started from notebooks/ instead of repo root — adjust
    ROOT = ROOT.parent
    sys.path.insert(0, str(ROOT / "src"))

print(f"Repo root: {ROOT}")
print(f"Dataset present: {(ROOT / 'data' / 'spotify_top50_songs_features.csv').is_file()}")
print(f"Artifacts present: {(ROOT / 'artifacts' / 'metrics.json').is_file()}")

Repo root: /Users/jayhuang/Desktop/INFO5368_PAML/INFO_5368_Final_Project
Dataset present: True
Artifacts present: True


## 1. Load the dataset and confirm shape

The dataset is `data/spotify_top50_songs_features.csv` — one row per (song, week) pair from the Spotify Weekly Top 50 Global chart.

In [2]:
df = pd.read_csv(ROOT / "data" / "spotify_top50_songs_features.csv")
df["hit"] = (df["rank"] <= 10).astype(int)

print(f"Rows: {len(df):,}")
print(f"Date range: {df['date'].min()}  →  {df['date'].max()}")
print(f"Artists: {df['primary_artist'].nunique():,}")
print(f"Class balance (hit=1): {df['hit'].mean():.1%}")
df.head(3)

Rows: 13,399
Date range: 2021-01-07  →  2026-02-19
Artists: 425
Class balance (hit=1): 20.0%


,rank,uri,artist_names,track_name,source,peak_rank,previous_rank,weeks_on_chart,streams,date,...,mfcc_2,chroma_mean,chroma_std,artist_list,feature_count,primary_artist,primary_artist_popularity,max_artist_popularity,monthly_listeners,hit
0,1,spotify:track:4MzXwWMhyBbmu6hOcLVD49,"Bad Bunny, JHAYCO",DÁKITI,Rimas Entertainment LLC,1,1,10,31544941,2021-01-07,...,103.523415,0.353403,0.298455,"['Bad Bunny', 'JHAYCO']",2,Bad Bunny,100,100,106142820.0,1
1,2,spotify:track:3tjFYV6RSFtuktYl3ZtYcq,"24kGoldn, iann dior",Mood (feat. iann dior),Records/Columbia,1,2,22,23461354,2021-01-07,...,93.771202,0.336845,0.299802,"['24kGoldn', 'iann dior']",2,24kGoldn,68,68,10207152.0,1
2,3,spotify:track:0VjIjW4GlUZAMYd2vXMi3b,The Weeknd,Blinding Lights,Republic Records,1,5,58,22981042,2021-01-07,...,109.673134,0.465111,0.293015,['The Weeknd'],1,The Weeknd,94,94,115833045.0,1


## 2. Load the saved metrics (Tables I, II, III)

These are the numbers reported in the paper. They were produced by running `python src/train.py` from the repo root. The notebook loads them from `artifacts/metrics.json` so the report and the notebook are guaranteed to be consistent.

In [3]:
metrics = json.loads((ROOT / "artifacts" / "metrics.json").read_text())

def _row(name, m):
    return {"Model": name, "Accuracy": m["accuracy"], "F1": m["f1"], "AUC": m["auc"]}

print("Table I — Stratified 80/20 split (production features)")
display(pd.DataFrame([
    _row("Logistic Regression", metrics["logistic_regression"]),
    _row("ANN (deployed)",       metrics["ann"]),
]))

print("\nTable II — Temporal split (train pre-2025-01-01, test on/after)")
ts = metrics["temporal_split"]
display(pd.DataFrame([
    _row("Logistic Regression", ts["logistic_regression"]),
    _row("ANN",                  ts["ann"]),
]))

print("\nTable III — Chart-context benchmark (+ streams, weeks_on_chart)")
bc = metrics["benchmark_chart_context"]
display(pd.DataFrame([
    _row("Logistic Regression (benchmark)", bc["logistic_regression"]),
    _row("ANN (benchmark)",                  bc["ann"]),
]))

print(f"\nComposite score — LR: {metrics['composite']['logistic_regression']:.4f}, "
      f"ANN: {metrics['composite']['ann']:.4f}")
print(f"Best deployed model: {metrics['deployment_model']}")

Table I — Stratified 80/20 split (production features)


,Model,Accuracy,F1,AUC
0,Logistic Regression,0.377501,0.352572,0.548615
1,ANN (deployed),0.707059,0.444921,0.716073



Table II — Temporal split (train pre-2025-01-01, test on/after)


,Model,Accuracy,F1,AUC
0,Logistic Regression,0.306157,0.354829,0.529701
1,ANN,0.645129,0.359194,0.594318



Table III — Chart-context benchmark (+ streams, weeks_on_chart)


,Model,Accuracy,F1,AUC
0,Logistic Regression (benchmark),0.925255,0.817006,0.966394
1,ANN (benchmark),0.925632,0.826432,0.972122



Composite score — LR: 0.4360, ANN: 0.6058
Best deployed model: ann


## 3. Feature-block ablation (Table IV)

Loaded from `artifacts/ablation_metrics.json`. To regenerate, run:

```bash
python src/eval_ablation.py --out-json artifacts/ablation_metrics.json
```

In [4]:
abl = json.loads((ROOT / "artifacts" / "ablation_metrics.json").read_text())
rows = []
for regime_name, payload in abl["regimes"].items():
    rows.append({
        "Regime": regime_name,
        "Features": payload["input_dim"],
        "LR Acc":  payload["logistic_regression"]["accuracy"],
        "LR F1":   payload["logistic_regression"]["f1"],
        "LR AUC":  payload["logistic_regression"]["auc"],
        "ANN Acc": payload["ann"]["accuracy"],
        "ANN F1":  payload["ann"]["f1"],
        "ANN AUC": payload["ann"]["auc"],
    })
print("Table IV — Feature-block ablation")
display(pd.DataFrame(rows).round(4))

Table IV — Feature-block ablation


,Regime,Features,LR Acc,LR F1,LR AUC,ANN Acc,ANN F1,ANN AUC
0,full,12,0.4575,0.3423,0.5470,0.4923,0.3911,0.6668
1,audio_only,9,0.2027,0.3350,0.5237,0.4522,0.3807,0.6529
2,artist_only,3,0.3484,0.3452,0.5462,0.4247,0.3553,0.6022


## 4. Optional: re-run experiments end to end

The cells above load saved numbers, which is what the paper reports. If you want to regenerate from scratch (5–15 minutes on a laptop), uncomment the cells below.

> ⚠️ Re-running `train.py` will overwrite `artifacts/*.npz` and `artifacts/metrics.json`. ANN runs are stochastic (mini-batch shuffling, dropout, He init), so the numbers will be within ~0.01–0.05 AUC of those in the paper but will not match exactly.

In [5]:
# Uncomment to regenerate everything from scratch:

# import subprocess
# subprocess.run(["python", str(ROOT / "src" / "train.py")], cwd=ROOT, check=True)
# subprocess.run(["python", str(ROOT / "src" / "eval_ablation.py"),
#                 "--out-json", "artifacts/ablation_metrics.json"], cwd=ROOT, check=True)
# subprocess.run(["python", str(ROOT / "src" / "eval_artist_stratified.py"),
#                 "--k", "5"], cwd=ROOT, check=True)

## 5. Artist-stratified evaluation (§III.D paragraph)

The script `src/eval_artist_stratified.py` runs 5-fold cross-validation grouped by `primary_artist`, so every test fold contains only artists the model has not seen during training. The headline numbers in the paper:

- **ANN**: AUC 0.492 ± 0.039,  F1 0.236 ± 0.035
- **LR**:  AUC 0.474 ± 0.032,  F1 0.310 ± 0.038

To regenerate, run:

```bash
python src/eval_artist_stratified.py --k 5
```

This takes ~5–10 minutes on a laptop. The output is printed to the terminal (no JSON file is written by default).